In [29]:
df = spark.table("bronze.bronze_azdo_workitems")

StatementMeta(, bcf2808c-91f9-4fda-90c3-e82904d8ef93, 31, Finished, Available, Finished, False)

In [30]:
display(df.limit(5))
df.printSchema()
print("Bronze Row Count: ", df.count())

StatementMeta(, bcf2808c-91f9-4fda-90c3-e82904d8ef93, 32, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 56b8f662-bebc-46b5-aacc-529ff1cd29f9)

root
 |-- organization_id: string (nullable = true)
 |-- organization_name: string (nullable = true)
 |-- project_id: string (nullable = true)
 |-- project_name: string (nullable = true)
 |-- work_item_id: integer (nullable = true)
 |-- activated_date: string (nullable = true)
 |-- area_path: string (nullable = true)
 |-- area_level_1: string (nullable = true)
 |-- area_level_2: string (nullable = true)
 |-- area_level_3: string (nullable = true)
 |-- area_level_4: string (nullable = true)
 |-- area_level_5: string (nullable = true)
 |-- assigned_to: string (nullable = true)
 |-- changed_date: timestamp (nullable = true)
 |-- closed_date: string (nullable = true)
 |-- created_by: string (nullable = true)
 |-- created_date: timestamp (nullable = true)
 |-- bug_aging_days: string (nullable = true)
 |-- p1_bug_aging_days: string (nullable = true)
 |-- p2_bug_aging_days: string (nullable = true)
 |-- cycle_time_days: string (nullable = true)
 |-- iteration_level_1: string (nullable = true)

In [31]:
from pyspark.sql.functions import *
from pyspark.sql.window import Window

ws = Window.partitionBy("work_item_id").orderBy(col("changed_date").desc()) 

df = df.withColumn(
    "row_num", 
    row_number().over(ws)
    )

df = df.filter(col("row_num") == 1)\
    .drop("row_num")

StatementMeta(, bcf2808c-91f9-4fda-90c3-e82904d8ef93, 33, Finished, Available, Finished, False)

In [32]:
df = df.withColumn("work_item_type", trim(col("work_item_type")))

StatementMeta(, bcf2808c-91f9-4fda-90c3-e82904d8ef93, 34, Finished, Available, Finished, False)

In [33]:
df = df.withColumn(
    "is_bug", 
    when(
        col("work_item_type") == "Bug",
        1
     ).otherwise(0)
)

StatementMeta(, bcf2808c-91f9-4fda-90c3-e82904d8ef93, 35, Finished, Available, Finished, False)

In [34]:
df = df.withColumn(
    "product_name",
    when(
        col("product_name").isNull() |
        (trim(col("product_name")) == ""),
        "Unknown Product"
        ).otherwise(trim(col("product_name"))) 
)

StatementMeta(, bcf2808c-91f9-4fda-90c3-e82904d8ef93, 36, Finished, Available, Finished, False)

In [35]:
df = df.withColumn(
    "reporting_month",
    date_format(col("created_date"), "yyyy-MM")
)

StatementMeta(, bcf2808c-91f9-4fda-90c3-e82904d8ef93, 37, Finished, Available, Finished, False)

In [36]:
df_count = df.count()
print("silver df Count: ", df_count)

StatementMeta(, bcf2808c-91f9-4fda-90c3-e82904d8ef93, 38, Finished, Available, Finished, False)

silver df Count:  499


In [37]:
df_final = df.select(
    "work_item_id",
    "product_name",
    "product_id",
    "work_item_type",
    "is_bug",
    "created_date",
    "changed_date",
    "reporting_month",
    "state",
    "ingestion_timestamp",
    "source_file_name"
)

StatementMeta(, bcf2808c-91f9-4fda-90c3-e82904d8ef93, 39, Finished, Available, Finished, False)

In [40]:
df_final.write.mode("overwrite")\
    .option("mergeSchema", "true")\
    .saveAsTable("silver.silver_workitems")

StatementMeta(, bcf2808c-91f9-4fda-90c3-e82904d8ef93, 42, Finished, Available, Finished, False)